# Automating Curriculum Design with AI: A Practical Guide Using AutoGen AgentChat

This notebook demonstrates how to build and orchestrate a multi-agent AI team using **Microsoft AutoGen 0.7.5 AgentChat** to research, structure, and refine high-quality educational courses.

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the difference between single-prompt LLMs and multi-agent systems
- Design and implement specialized AI agents (SME, Instructional Designer, QA Reviewer)
- Orchestrate agent collaboration using AutoGen's Team orchestration
- Generate complete, pedagogically sound curricula in minutes

## Introduction

### The Challenge
Designing comprehensive curricula is traditionally resource-intensive, requiring SMEs, instructional designers, and QA reviewers spending weeks on syllabi, learning objectives, and assessments. By the time content is ready, the industry has often moved on.

### The Solution: Multi-Agent Systems
Instead of relying on a single LLM (which produces generic, shallow content), we build specialized agents that:
- **Debate and refine** each other's work
- **Apply domain expertise** (pedagogy, subject matter, quality assurance)
- **Iterate autonomously** until consensus is reached

### Why AutoGen AgentChat?
Microsoft AutoGen 0.7.5+ is a modern framework that enables:
- Multiple conversing AI agents with distinct personas
- Flexible team orchestration patterns (Round Robin, Selector, etc.)
- Built-in tool support and human-in-the-loop interactions

## The Multi-Agent Architecture

Our AI curriculum team consists of four specialized agents:

| Agent | Role | Focus |
|-------|------|-------|
| **Subject Matter Expert** | Brain trust | Core concepts, industry standards, technical accuracy |
| **Instructional Designer** | Learning architect | Bloom's Taxonomy, cognitive load, assessments |
| **QA Reviewer** | Critical eye | Flow, prerequisites, jargon, difficulty curve |
| **Human Proxy** | Human-in-the-loop | Goal setting, constraints, final approval |

### Key Benefits
- **Speed**: Months of work → minutes
- **Scalability**: Generate customized curricula at scale
- **Quality**: Simulated diverse perspectives before real classroom use
- **Control**: Human oversight at key milestones

## Prerequisites & Setup

First, ensure you have the required packages installed in your virtual environment:

In [1]:
# Install AutoGen AgentChat (run in your terminal if not already installed)
# pip install autogen-agentchat autogen-ext[openai]

import os
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

print("AutoGen AgentChat imported successfully!")
print("Required modules loaded:")
print("  - AssistantAgent for creating specialized agents")
print("  - RoundRobinGroupChat for team orchestration")
print("  - Console for displaying agent conversations")
print("  - OpenAIChatCompletionClient for LLM integration")

AutoGen AgentChat imported successfully!
Required modules loaded:
  - AssistantAgent for creating specialized agents
  - RoundRobinGroupChat for team orchestration
  - Console for displaying agent conversations
  - OpenAIChatCompletionClient for LLM integration


### Set Your API Key

You need an OpenAI API key. Set it as an environment variable:

In [ ]:
# Set your OpenAI API key
# Option 1: Set as environment variable before running this notebook
# export OPENAI_API_KEY='your-key-here'

# Option 2: Set directly (not recommended for production)
os.environ["OPENAI_API_KEY"] = "Set your OpenAI API key"

# Verify the key is set
api_key = os.environ.get("OPENAI_API_KEY")
if api_key:
    print("API key is set (starts with:", api_key[:7], "...)")
else:
    print("WARNING: OPENAI_API_KEY not found in environment variables!")
    print("Please set it using: export OPENAI_API_KEY='your-key-here'")

API key is set (starts with: sk-proj ...)


## Step 1: Define LLM Configuration

Configure the language model client for all agents:

In [3]:
# Define the model client
# You can use different models: gpt-4o, gpt-4o-mini, o3-mini, etc.
model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    # api_key is automatically read from OPENAI_API_KEY environment variable
    temperature=0.2,  # Low temperature for factual accuracy
)

print("Model client configured!")
print("Model: gpt-4o")
print("Temperature: 0.2")

Model client configured!
Model: gpt-4o
Temperature: 0.2


## Step 2: Initialize the Agents

Create our four specialized agents with distinct system messages.

### Agent 1: Subject Matter Expert

The SME focuses solely on **what** is taught—providing accurate, comprehensive knowledge without worrying about formatting.

In [4]:
sme_agent = AssistantAgent(
    name="Subject_Matter_Expert",
    model_client=model_client,
    system_message="""You are a brilliant Subject Matter Expert.
    
Your goal is to provide accurate, up-to-date, and comprehensive facts, concepts, and industry standards for the requested topic.
    
Focus on:
- Core concepts and terminology
- Latest industry trends and standards
- Technical accuracy and depth
    
Do NOT worry about:
- Formatting the curriculum
- Pedagogical structure
- Difficulty progression
    
Just provide the raw knowledge, key terminologies, and latest trends.
    
Keep your responses concise but comprehensive.""",
)

print("Created:", sme_agent.name)

Created: Subject_Matter_Expert


### Agent 2: Instructional Designer

The ID focuses on **how** the content is taught—applying pedagogical frameworks and managing cognitive load.

In [5]:
id_agent = AssistantAgent(
    name="Instructional_Designer",
    model_client=model_client,
    system_message="""You are an expert Instructional Designer.
    
Your goal is to take raw concepts from the SME and structure them into a pedagogical format.
    
Always:
- Create week-by-week modules
- Apply Bloom's Taxonomy to learning objectives (e.g., "By the end of this module, learners will be able to...")
- Define clear learning objectives for each module
- Suggest practical, relevant assessments
- Manage cognitive load appropriately for the target audience
- Sequence topics for logical difficulty progression
    
Format your output clearly with:
- Module numbers and titles
- Learning objectives
- Topics covered
- Assessment activities
    
Keep your responses structured and easy to read.""",
)

print("Created:", id_agent.name)

Created: Instructional_Designer


### Agent 3: QA Reviewer

The Reviewer acts as a critical eye and student advocate, ensuring accessibility and logical flow.

In [6]:
reviewer_agent = AssistantAgent(
    name="QA_Reviewer",
    model_client=model_client,
    system_message="""You are a strict QA Reviewer and Student Advocate.
    
Your goal is to review the curriculum and ensure it meets the target audience's needs.
    
Always look for:
- Flaws in logical flow
- Steep learning curves
- Missing prerequisites
- Undefined jargon or terminology
- Misalignment between content and audience skill level
- Unrealistic assessments or projects
    
Push back on the SME and Instructional Designer until the curriculum is accessible and logically flows.
Be constructive but thorough in your critiques.
    
Once satisfied, respond with "CURRICULUM APPROVED" and provide the final polished curriculum.
    
Keep your feedback specific and actionable.""",
)

print("Created:", reviewer_agent.name)

Created: QA_Reviewer


## Step 3: Create the Team

Now we place all agents in a team where they can collaborate. In AutoGen AgentChat, we use `RoundRobinGroupChat` which cycles through agents in order, allowing each to respond based on the conversation history.

In [12]:
# Create the team with our agents
# The agents will take turns in the order specified
curriculum_team = RoundRobinGroupChat(
    participants=[sme_agent, id_agent, reviewer_agent],
    max_turns=6,  # Limit conversation to 6 total turns
)

print("Team created successfully!")
print("Participants:")
print("  1. Subject_Matter_Expert")
print("  2. Instructional_Designer")
print("  3. QA_Reviewer")
print("\nMax turns: 15")
print("\nThe team will cycle through these agents in order.")

Team created successfully!
Participants:
  1. Subject_Matter_Expert
  2. Instructional_Designer
  3. QA_Reviewer

Max turns: 15

The team will cycle through these agents in order.


## Step 4: Define Your Curriculum Task

Customize the task description below based on your needs:

In [13]:
# Customize your curriculum request here
task_description = """Create a 4-week introductory curriculum on 'Prompt Engineering for Marketing Professionals'.

Target Audience:
- Marketing professionals with ZERO coding background
- Understand basic marketing concepts
- Want to use AI for content creation, campaign ideation, and brand consistency

Requirements:
- 4 weeks of content
- Clear learning objectives for each week
- Practical, hands-on assessments
- Progress from basic to advanced concepts
- Include real-world marketing examples

Please debate and refine the curriculum until perfect.
End with a complete, structured syllabus that a human instructor could use immediately."""

print("Task defined! Ready to generate curriculum.")
print("\nTask Description:")
print(task_description)

Task defined! Ready to generate curriculum.

Task Description:
Create a 4-week introductory curriculum on 'Prompt Engineering for Marketing Professionals'.

Target Audience:
- Marketing professionals with ZERO coding background
- Understand basic marketing concepts
- Want to use AI for content creation, campaign ideation, and brand consistency

Requirements:
- 4 weeks of content
- Clear learning objectives for each week
- Practical, hands-on assessments
- Progress from basic to advanced concepts
- Include real-world marketing examples

Please debate and refine the curriculum until perfect.
End with a complete, structured syllabus that a human instructor could use immediately.


## Step 5: Launch the Multi-Agent Curriculum Generation

Run the cell below to start the autonomous curriculum generation. Watch as the agents collaborate in real-time!

### What to Expect:
1. **SME** will dump comprehensive technical knowledge
2. **ID** will structure it into weekly modules
3. **QA** will critique and suggest improvements
4. The cycle continues until consensus is reached or max_turns is reached

In [14]:
# Helper function to run the team and display results
async def generate_curriculum():
    """Run the curriculum generation team and display the conversation."""
    print("=" * 80)
    print("CURRICULUM GENERATION STARTED")
    print("=" * 80)
    print()
    
    # Run the team with the task
    result = await curriculum_team.run(task=task_description)
    
    print()
    print("=" * 80)
    print("CURRICULUM GENERATION COMPLETED")
    print("=" * 80)
    
    return result

# Run the generation
result = await generate_curriculum()

CURRICULUM GENERATION STARTED


CURRICULUM GENERATION COMPLETED


## View the Final Result

After the team completes their work, let's extract and display the final curriculum:

In [15]:
# Display the final messages from the team
print("\n" + "=" * 80)
print("FINAL CURRICULUM OUTPUT")
print("=" * 80 + "\n")

# Iterate through the result to show the conversation
for i, message in enumerate(result.messages, 1):
    print(f"[Message {i}] - Source: {message.source}")
    print(f"{message.content}")
    print("-" * 80)

# The last message from QA_Reviewer should contain the final approved curriculum
final_msg = result.messages[-1]
if final_msg.source == "QA_Reviewer" and "APPROVED" in final_msg.content:
    print("\n🎉 CURRICULUM APPROVED BY QA REVIEWER! 🎉\n")
    print(final_msg.content)


FINAL CURRICULUM OUTPUT

[Message 1] - Source: user
Create a 4-week introductory curriculum on 'Prompt Engineering for Marketing Professionals'.

Target Audience:
- Marketing professionals with ZERO coding background
- Understand basic marketing concepts
- Want to use AI for content creation, campaign ideation, and brand consistency

Requirements:
- 4 weeks of content
- Clear learning objectives for each week
- Practical, hands-on assessments
- Progress from basic to advanced concepts
- Include real-world marketing examples

Please debate and refine the curriculum until perfect.
End with a complete, structured syllabus that a human instructor could use immediately.
--------------------------------------------------------------------------------
[Message 2] - Source: Subject_Matter_Expert
**Week 1: Foundations of AI and Prompt Engineering in Marketing**

**Learning Objectives:**
- Grasp the fundamentals of AI and its role in marketing.
- Understand the concept and significance of promp

## Expected Output Example

After the agents reach consensus, you'll see output similar to this:

```
Course: Prompt Engineering for Marketing Professionals
Duration: 4 Weeks | Audience: Marketing Professionals (No-code)

---

### Week 1: Fundamentals of Generative AI for Marketers

**Learning Objective:**
By the end of this week, learners will be able to define what an LLM is and construct basic, single-turn prompts for content ideation.

**Topics Covered:**
- Introduction to LLMs
- The Anatomy of a Good Prompt (Role, Task, Context, Format)
- Zero-Shot Prompting

**Assessment:**
Project Ideation: Use an LLM to generate 10 blog post titles based on a provided product brief.

---

### Week 2: Advanced Prompt Structures for Brand Consistency

**Learning Objective:**
By the end of this week, learners will be able to apply few-shot prompting to match brand voice and tone.

**Topics Covered:**
- Few-Shot Prompting
- Providing Examples in Context
- Formatting Outputs (Markdown, CSV for content calendars)

**Assessment:**
Tone-Matching Exercise: Provide the LLM with 3 previous social media posts and prompt it to write a 4th post in the exact same brand voice.

---

(...Weeks 3 and 4 follow with similar structure...)
```

This output is a pedagogically sound framework ready for immediate use!

## Customization Options

### Try Different Topics
Modify the task_description variable to generate curricula for other subjects:

In [ ]:
# Example alternative topics (uncomment to try):

# Python for Data Analysis (8 weeks, business analysts)
# task_description = """Create an 8-week curriculum on 'Python for Data Analysis'.
# Audience: Business analysts with basic Excel knowledge, no coding experience.
# Focus on pandas, data visualization, and business reporting."""

# Sustainability in Supply Chain (6 weeks, operations managers)
# task_description = """Create a 6-week curriculum on 'Sustainability in Supply Chain Management'.
# Audience: Operations managers familiar with logistics but new to ESG concepts.
# Include carbon footprint calculation, sustainable sourcing, and reporting frameworks."""

# Digital Marketing Strategy (10 weeks, small business owners)
# task_description = """Create a 10-week curriculum on 'Digital Marketing Strategy for Small Business'.
# Audience: Small business owners with limited marketing budget and technical skills.
# Cover SEO, social media, email marketing, and analytics."""

print("Choose a topic above, uncomment, and run the curriculum generation cell again!")

## Advanced: Using Different Team Patterns

AutoGen AgentChat supports various team orchestration patterns. Here are some alternatives to RoundRobinGroupChat:

In [ ]:
# Example: Using SelectorTeam for dynamic speaker selection
from autogen_agentchat.teams import SelectorTeam
from autogen_agentchat.conditions import MaxMessageTermination

# Create a selector team where an LLM chooses the next speaker
# This is more expensive but can lead to more natural conversations

# selector_team = SelectorTeam(
#     participants=[sme_agent, id_agent, reviewer_agent],
#     model_client=model_client,  # Uses LLM to select next speaker
#     termination_condition=MaxMessageTermination(max_messages=20),
# )

print("SelectorTeam example available (commented out)")
print("SelectorTeam uses an LLM to dynamically select the next speaker based on context.")
print("This creates more natural conversations but costs more due to extra API calls.")

## Best Practices

### 1. Team Orchestration Patterns

| Pattern | Behavior | Use Case |
|--------|----------|----------|
| RoundRobinGroupChat | Cycles through agents in order | Deterministic flow, lower cost |
| SelectorTeam | LLM selects next speaker | Dynamic conversations, higher cost |

### 2. Cost Optimization

Use different models for different agents:

```python
# Expensive model for critical thinking
sme_model = OpenAIChatCompletionClient(model="gpt-4o", temperature=0.2)
reviewer_model = OpenAIChatCompletionClient(model="gpt-4o", temperature=0.1)

# Cheaper model for formatting/structure
id_model = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=0.3)

# Assign to agents
sme_agent = AssistantAgent(name="SME", model_client=sme_model, ...)
id_agent = AssistantAgent(name="ID", model_client=id_model, ...)
reviewer_agent = AssistantAgent(name="QA", model_client=reviewer_model, ...)
```

### 3. Termination Conditions

Control when the team stops working:

```python
from autogen_agentchat.conditions import MaxMessageTermination, TextTermination

# Stop after N messages
# team = RoundRobinGroupChat(..., termination=MaxMessageTermination(20))

# Stop when specific text appears
# team = RoundRobinGroupChat(..., termination=TextTermination("CURRICULUM APPROVED"))
```

## Troubleshooting

| Issue | Solution |
|-------|----------|
| Agents talking forever | Reduce max_turns or add termination condition |
| Generic output | Strengthen system messages with specific constraints |
| Too much jargon | Emphasize target audience in task description |
| Shallow content | Increase temperature slightly for more creativity |
| API errors | Check API key, rate limits, and model availability |
| Agents not debating | Make QA Reviewer's system message more strict |
| Import errors | Ensure autogen-agentchat and autogen-ext[openai] are installed |

## Summary

In this notebook, you learned how to:

1. **Design specialized agents** with distinct roles (SME, Instructional Designer, QA Reviewer)
2. **Orchestrate multi-agent collaboration** using AutoGen's RoundRobinGroupChat
3. **Generate complete curricula** through autonomous agent debate and refinement
4. **Customize for your needs** by modifying task descriptions and agent configurations

### Next Steps

- Experiment with different topics and audience profiles
- Try different team orchestration patterns (SelectorTeam)
- Add tool support for external data retrieval
- Integrate with your LMS or content management system

### Resources

- [AutoGen AgentChat Documentation](https://microsoft.github.io/autogen/dev/user-guide/agentchat-user-guide/index.html)
- [AutoGen GitHub](https://github.com/microsoft/autogen)
- [Bloom's Taxonomy](https://cft.vanderbilt.edu/guides-sub-pages/blooms-taxonomy/)

Happy curriculum building!